# 02 — Agent description pre-summarization

**Goal**: collapse each agent's `description` (often 200-2000 chars of marketing copy) into ONE compact sentence (≤ 30 words) describing industry + main service. Persist into `agents.agentSummary` so all later classification notebooks read compact agent context instead of re-summarizing.

Why: the current Go pipeline either pastes the full description (token-heavy, blows past 3B context) or truncates it (loses the punchline). A pre-cached summary fixes both.

**Scope**: this notebook only prepares context for classification across both benchmark sources: `rule_based` and `hand_labelled`. The final AI classifier outputs exactly four real categories: `junk`, `service_feedback`, `config_feedback`, and `app_specific`. `others` is only the rule-based fallback/source bucket for rows the rules could not cover; those rows are handed to the LLM to classify into one of the four real categories.

**Cost**: one Ollama call per unique agent that lacks a fresh cached summary.

**Idempotent**: rows where `agentSummary` already exists and `description` is unchanged are skipped (uses `descriptionHash` to detect changes).

In [1]:
import sys, hashlib, json
from pathlib import Path
AI_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(AI_ROOT))

import pandas as pd
from tqdm.auto import tqdm
from shared.mongo_client import agents_coll
from shared.ollama_client import OllamaClient
from shared.prompts import AGENT_SUMMARY_SYSTEM, agent_summary_user_msg
from shared.context_builder import format_services

SPLITS_DIR = AI_ROOT / 'data' / 'splits'
SPLIT_FILES = [
    SPLITS_DIR / 'rule_based' / 'train.parquet',
    SPLITS_DIR / 'rule_based' / 'val.parquet',
    SPLITS_DIR / 'rule_based' / 'test.parquet',
    SPLITS_DIR / 'hand_labelled' / 'test.parquet',
]
splits = pd.concat([pd.read_parquet(p) for p in SPLIT_FILES if p.exists()], ignore_index=True)
print('total rows:', len(splits))
print('by bench:', splits['bench'].value_counts().to_dict() if 'bench' in splits else 'n/a')

needed = splits[['chain_id','agent_id']].drop_duplicates().to_records(index=False).tolist()
print('unique agents:', len(needed))

total rows: 7012
by bench: {'rule_based': 6815, 'hand_labelled': 197}
unique agents: 3977


In [2]:
def desc_hash(desc: str) -> str:
    return hashlib.sha1((desc or '').encode('utf-8')).hexdigest()[:16]

# Fetch agent docs in bulk.
ids = [f'{c}:{a}' for c, a in needed]
docs = list(agents_coll().find({'_id': {'$in': ids}}))
by_id = {d['_id']: d for d in docs}
print('agent docs found:', len(by_id), '/', len(ids))

todo = []
for d in docs:
    desc = d.get('description', '') or ''
    if not desc.strip():
        continue
    h = desc_hash(desc)
    if d.get('agentSummary') and d.get('descriptionHash') == h:
        continue
    todo.append(d)
print('to summarize:', len(todo))

agent docs found: 3946 / 3977
to summarize: 2039


In [3]:
# Run summarization. Uses qwen2.5:3b — small + fast + good enough at one-sentence summaries.
MODEL = 'qwen2.5:3b'
summaries: dict[str, tuple[str, str]] = {}  # _id -> (summary, hash)

with OllamaClient(model=MODEL) as cli:
    if not cli.health():
        raise RuntimeError(
            f'Ollama not reachable at {cli.base_url}. '
            f'Run `ollama serve` on the host and `ollama pull {MODEL}` first.'
        )
    for d in tqdm(todo):
        desc = d.get('description', '') or ''
        services = format_services(d.get('services', []))
        msg = agent_summary_user_msg(d.get('name', ''), desc, services)
        out = cli.summarize(AGENT_SUMMARY_SYSTEM, msg, num_predict=96)
        # Strip surrounding quotes and trailing periods Ollama sometimes adds.
        out = out.strip().strip('"').strip("'").strip()
        if out:
            summaries[d['_id']] = (out, desc_hash(desc))

print('summarized:', len(summaries))
list(summaries.items())[:3]

  0%|          | 0/2039 [00:00<?, ?it/s]

summarized: 2039


[('1:10289',
  ('KIZUNA is an ERC-8004 trust agent verifying identities and behaviors, executing autonomous transactions in DeFi ecosystem.',
   'f5197c94bd8e77db')),
 ('1:10303',
  ('DeFi: Finds and analyzes academic papers for users.', '53a68bebdc3193f1')),
 ('1:10307',
  ('DeFi - Analyzes data to make & execute financial decisions on-chain.',
   '5dc0d435374f9836'))]

In [4]:
# Persist to Mongo. One bulk update.
from pymongo import UpdateOne
ops = [
    UpdateOne({'_id': _id}, {'$set': {'agentSummary': s, 'descriptionHash': h}})
    for _id, (s, h) in summaries.items()
]
if ops:
    r = agents_coll().bulk_write(ops, ordered=False)
    print('modified:', r.modified_count, '/ matched:', r.matched_count)
else:
    print('nothing to write')

modified: 2039 / matched: 2039


In [5]:
# Spot-check 5 random summaries
import random
for _id, (s, _) in random.sample(list(summaries.items()), min(5, len(summaries))):
    d = by_id[_id]
    print(f'\n— {d.get("name","?")} ({_id})')
    print('  desc :', (d.get('description','') or '')[:140].replace('\n',' '))
    print('  summ :', s)


— Story Scoring Agent (1:14645)
  desc : Scores user stories (0-100). Score ≥60 grants one claim of 100 Story tokens. Pay 10 USDC to claim. Backed by 8004mint.com. Website: https://
  summ : DeFi: Scores user stories for Story tokens, grants 1 claim of 100 STORY for ≥60 score, pay 10 USDC. Backed by 8004mint.com.

— jawapro (8453:22192)
  desc : Jawapro is an intelligent AI agent designed to deliver advanced data analysis, automated reporting, and real-time insights across Web2 and W
  summ : Jawapro is an AI agent for Web2/Web3 that automates data analysis, analytics, and insights with real-time charting and workflow automation.

— HodlAI Protocol (56:96)
  desc : The first agent-native compute protocol. Hold $HODLAI to access LLMs. Visit hodlai.fun
  summ : DeFi: Access AI models by staking HODLAI tokens.

— ₿ (8453:14413)
  desc : This autonomous agent provides structured data analysis and informational services related to gold as a financial and physical asset. The ag
  summ : DeF